# Neural Networks

In [41]:
# Import necessary Python libraries
import numpy as np

## Components of a Neural Network
Forward Propagation
- Dense Layer
- Activation Function (sigmoid function)
- Loss/error calculation (Mean Squared Error - MSE)

Backward Propagation

## Forward Propagation

### Output Sum Calculation (Dense Layer)
$$y = w_{ij} \times x_{j} + b_{j}$$
$$Y = W \cdot X + B$$

In [42]:
def sum(input: np.ndarray, weight: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.dot(weight, input) + bias

### Activation Function
#### Sigmoid Function
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

In [43]:
def sigmoid(x) -> float:
    return 1 / (1 + np.exp(-x))

### Loss/Error Function
#### Mean Squared Error (MSE)
$$E = \sum_{i}\frac{(Y_{i, true} - Y_{i, pred})^2}{No. Outputs}$$

#### Derivative of Mean Squared Error
$$\frac{dE}{dY} = \frac{2 (Y_{i, pred} - Y_{i, true})}{No. Outputs}$$

In [44]:
# Function for mean squared error
def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.power((y_true - y_pred), 2))

# Function for the derivative of the mean squared error function
def mse_prime(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return 2 * (y_true - y_pred) / np.size(y_true)

## Backward Propagation

### Gradient Descent

#### Weight Gradient
E.g. for weight at w11
$$
\begin{align*}
\frac{dE}{dW} &= \frac{dE}{dY} \frac{dY}{dW} \\
              &= \frac{dE}{dy_{1}} \frac{dy_{1}}{dw_{11}} + \frac{dE}{dy_{2}} \frac{dy_{2}}{dw_{11}} + \cdots + \frac{dE}{dy_{j}} \frac{dy_{j}}{dw_{11}} \\
              &= \frac{dE}{dy_{1}} \cdot x_{1} + 0 + \cdots + 0 \\
              &= \frac{dE}{dy_{j}} \cdot x_{i}
\end{align*}
$$

$$\therefore \frac{dE}{dw_{ji}} = \frac{dE}{dY} \cdot X^{T}$$

### Bias Gradient
E.g. for bias at b1
$$
\begin{align*}
\frac{dE}{dB} &= \frac{dE}{dY} \frac{dY}{dB} \\
              &= \frac{dE}{dy_{1}} \frac{dy_{1}}{db_{1}} + \cdot + \frac{dE}{dy_{j}} \frac{dy_{j}}{db_{j}} \\
              &= \frac{dE}{dy_{j}} \cdot 1 \\
              &= \frac{dE}{dY}
\end{align*}
$$

$$\therefore \frac{dE}{dB} = \frac{dE}{dY}$$

### Input Gradient (for hidden layers)
E.g. for input at x1
$$
\begin{align*}
\frac{dE}{dX} &= \frac{dE}{dY} \frac{dY}{dX} \\
              &= \frac{dE}{dy_{1}} \frac{dy_{1}}{dx_{1}} + \frac{dE}{dy_{2}} \frac{dy_{2}}{dw_{1}} + \cdots + \frac{dE}{dy_{j}} \frac{dy_{j}}{dw_{i}} \\
              &= \frac{dE}{dy_{1}} \cdot w_{11} + \frac{dE}{dy_{2}} \cdot w_{21} + \cdots + \frac{dE}{dy_{i}} \cdot w_{ji}
\end{align*}
$$

$$\therefore \frac{dE}{dx_{i}} =  W^{T} \cdot \frac{dE}{dy_{j}}$$

### Weight Update Formula
$$\Delta w_{ij} = Learning \space Rate \times Weight \space Gradient \times Output$$

### Bias Update Formula
$$\Delta b_{j} = Learning \space Rate \times Bias \space Gradient$$

In [45]:
# Backwards propagation (gradient descent) algorithm
def gradient_descent(
    output: np.ndarray, 
    output_gradient: np.ndarray, 
    learn_rate: float, 
    weight: np.ndarray, 
    bias: np.ndarray
) -> np.ndarray:

    weight_gradient = np.dot(output_gradient, output.T)
    bias_gradient = output_gradient

    next_output_gradient = np.dot(weight.T, output_gradient)

    weight -= learn_rate * weight_gradient
    
    bias -= learn_rate * bias_gradient

    return weight, bias, weight_gradient, bias_gradient, next_output_gradient

In [46]:
# main (XOR test)
X = np.reshape([[0, 0], [0, 1], [1, 0], [1, 1]], (4, 2, 1)) # Input array here
Y = np.reshape([[0], [1], [1], [0]], (4, 1, 1)) # True outputs here

# Network architecture
input_size = 2
hidden_size = 3
output_size = 1

w1 = np.random.randn(hidden_size, input_size)
b1 = np.random.randn(hidden_size, 1)

w2 = np.random.randn(output_size, hidden_size)
b2 = np.random.randn(output_size, 1)

epochs = 10000
learn_rate = 0.001

for e in range(epochs):
    total_error = 0
    for x, y in zip(X, Y):
        hidden_input = sum(x, w1, b1)
        hidden_output = sigmoid(hidden_input)

        output_input = sum(hidden_output, w2, b2)
        final_output = sigmoid(output_input)

        total_error += mse(y, final_output)

        dE_dY = mse_prime(y, final_output)

        output_gradient = dE_dY * final_output * (1 - final_output)
        w2, b2, dE_dW2, dE_dB2, dE_dHidden = gradient_descent(hidden_output, output_gradient, learn_rate, w2, b2)

        hidden_gradient = dE_dHidden * hidden_output * (1 - hidden_output)
        w1, b1, dE_dW1, dE_dB1, _ = gradient_descent(x, hidden_gradient, learn_rate, w1, b1)

    total_error /= len(x)
    if e % 1000 == 0:
        print(f"Epoch {e}/{epochs} - Error: {total_error:.6f}")

Epoch 0/10000 - Error: 0.733281
Epoch 1000/10000 - Error: 0.866019
Epoch 2000/10000 - Error: 0.918923
Epoch 3000/10000 - Error: 0.943685
Epoch 4000/10000 - Error: 0.957437
Epoch 5000/10000 - Error: 0.966027
Epoch 6000/10000 - Error: 0.971846
Epoch 7000/10000 - Error: 0.976026
Epoch 8000/10000 - Error: 0.979162
Epoch 9000/10000 - Error: 0.981596


## Object Oriented Programming Method

In [47]:
# file name: layer.py
class Layer:
    def __init__(self):
        self.input = None
        self.output = None

    def forward(self, input):
        pass
        
    def backward(self, input, learning_rate):
        pass

In [48]:
# file name: dense.py
# from layer import Layer

class Dense(Layer):
    def __init__(self, input_size: int, output_size: int):
        super().__init__()
        self.weight = np.random.randn(output_size, input_size)
        self.bias = np.random.randn(output_size, 1)

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.input = input
        # Z = WX + B
        return np.dot(self.weight, self.input) + self.bias

    def backward(self, output_gradient: np.ndarray, learn_rate: float) -> np.ndarray:
        weight_gradient = np.dot(output_gradient, self.input.T)
        bias_gradient = output_gradient

        self.weight -= learn_rate * weight_gradient
        self.bias -= learn_rate * bias_gradient

        return np.dot(self.weight.T, output_gradient)

In [49]:
# file name: activation.py
# from layer import Layer

class Activation(Layer):
    def __init__(self, activation, activation_prime):
        self.activation = activation
        self.activation_prime = activation_prime

    def forward(self, input):
        self.input = input
        return self.activation(self.input)

    def backward(self, output_gradient, learn_rate):
        return self.activation_prime(self.input)

In [50]:
# file name: activation-function.py
# from activation import Activation

class Sigmoid(Activation):
    def __init__(self):
        sigmoid = lambda x: 1 / (1 + np.exp(-x))
        sigmoid_prime = lambda y: sigmoid(y) * (1 - sigmoid(y))
        super().__init__(sigmoid, sigmoid_prime)

In [51]:
# file name: error.py
# Calculate error
def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.power(y_true - y_pred, 2))

# Derivative of mean squared error function (dE/dY)
def mse_prime(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return 2 * (y_pred - y_true) / np.size(y_true)

In [52]:
# file name: main.py
# import numpy as np

# from layer import Layer
# from dense import Dense
# from activation import Activation
# from activation_function import Sigmoid
# from error import mse, mse_prime

X = np.reshape([[0, 0], [0, 1], [1, 0], [1, 1]], (4, 2, 1)) # Input array here
Y = np.reshape([[0], [1], [1], [0]], (4, 1, 1)) # True outputs here

network = [
    Dense(2, 3), # Edit Dense parameters according to input data size
    Sigmoid(),
    Dense(3, 1),
    Sigmoid()
]

epochs = 10000
learn_rate = 0.1

for e in range(epochs):
    error = 0
    for x, y in zip(X, Y):
        # FORWARD PROPAGATION
        output = x
        for layer in network:
            output = layer.forward(output)

        error += mse(y, output)

        # BACKWARD PROPAGATION
        gradient = mse_prime(y, output)

        for layer in reversed(network):
            gradient = layer.backward(gradient, learn_rate)

    error /= len(x)
    if e % 1000 == 0:
        print(f"Epoch {e}/{epochs} - Error: {error:.6f}")

Epoch 0/10000 - Error: 0.741498
Epoch 1000/10000 - Error: 0.994940
Epoch 2000/10000 - Error: 0.997478
Epoch 3000/10000 - Error: 0.998322
Epoch 4000/10000 - Error: 0.998743
Epoch 5000/10000 - Error: 0.998995
Epoch 6000/10000 - Error: 0.999163
Epoch 7000/10000 - Error: 0.999283
Epoch 8000/10000 - Error: 0.999373
Epoch 9000/10000 - Error: 0.999443
